# 5. 其他拆分方式速记（不展开重复实例）

来源：《尚硅谷-10-RAG.pdf》，第 37–46 页；对应课件 **2.3.4 ③–⑦**。页码均指 PDF 阅读器中的页序。

整理方式：原理保留关键逻辑；第 2 章示例精简，重点学习拆分器源码与递归拆分。
只记录到 2.5.2，不收录 2.5.3 案例。

**运行约定**：在 `D:/grow/langchain/7_RAG` 打开本文件，选择
`D:/conda/envs/langchain/python.exe` 对应的 Python 内核；代码单元按本文件顺序运行。
各 Notebook 的变量相互独立。使用 `len` 的示例以 Python 字符长度计数，不能据此推断 Token 数。

## 5.1 一张表区分本节方法

| 方法 | 主要依据 | 适合记住的区别 |
| --- | --- | --- |
| `TokenTextSplitter` | Token 序列窗口 | 直接按 Token 分块；不理解句子，可能拆开中文字符的编码边界 |
| `CharacterTextSplitter.from_tiktoken_encoder()` | 用 Tokenizer 算长度，仍按一个分隔符拆分 | 改的是计数方式；不可再细分的片段仍可能超过预算 |
| `RecursiveCharacterTextSplitter.from_tiktoken_encoder()` | 用 Tokenizer 计数，结合递归细分 | 同时考虑边界和长度；Tokenizer 应匹配实际模型 |
| `SemanticChunker` | 句子 / 句子窗口的 Embedding 距离 | 语义差异大时设断点，需要额外嵌入计算 |
| `HTMLHeaderTextSplitter` | HTML 标题层级 | 保留 `h1`、`h2` 等标题到 metadata；不是 Token 长度控制器 |
| `RecursiveCharacterTextSplitter.from_language()` | 语言相关分隔符 | 课件「CodeTextSplitter」使用的实际入口；仍是规则拆分，不是 AST 解析器 |
| `MarkdownTextSplitter` | Markdown 常见语法边界 | 基于 Markdown 规则做递归字符拆分 |
| `MarkdownHeaderTextSplitter` | Markdown 标题层级 | 按标题组织 Document 和元数据，可再接递归拆分控制块长 |

主线仍是上一份递归拆分器；本文件只保留比较与参数速记，不再复制课件中的长输入和整屏输出。

## 5.2 Token 拆分：不要把字符数当 Token 数

模型上下文和费用通常以 Token 为单位；同一个字符串在不同 Tokenizer 下计数可能不同。
不能固定认为「一个汉字一个 Token」，也不能因为一次拆分恰好落在句末，就认为算法识别了句子。

`TokenTextSplitter` 的 `chunk_overlap` 单位是该编码器的 Token。
`from_tiktoken_encoder()` 是工厂方法，返回的仍是调用它的拆分器类型；
Character 与 Recursive 的拆分行为不会因为改用 Token 计数就变成一样。

对 Token 上限的检查还需要给提示词、问题、历史消息和生成回答预留预算。
本笔记不下载 Tokenizer 数据，也不把某个旧模型的窗口大小当作通用上限。

## 5.3 SemanticChunker：只记流程与关键参数

```text
正则拆句 → 组合邻近句子窗口 → 计算 Embedding
         → 相邻窗口余弦距离 → 阈值判定 → 将连续句子合成块
```

| 参数 | 作用 / 本地 0.4.1 的默认值 |
| --- | --- |
| `embeddings` | 提供向量模型；接口通常会调用 `embed_documents()` |
| `buffer_size` | 每个句子左右一起参与嵌入的邻居数量，默认 `1` |
| `breakpoint_threshold_type` | 阈值算法，默认 `"percentile"` |
| `breakpoint_threshold_amount` | 阈值量；各模式含义不同，不能混用百分比和倍数 |
| `sentence_split_regex` | 默认针对英文标点后空白：`r"(?<=[.?!])\s+"` |
| `min_chunk_size` | 可选最小块长约束，不是通用的最大 Token 限制 |

| 阈值类型 | 本地实现的判定基准 | 默认量 |
| --- | --- | --- |
| `percentile` | 相邻窗口距离分布的指定百分位 | `95` |
| `standard_deviation` | 平均距离 + N × 标准差 | `3` |
| `interquartile` | 平均距离 + N × 四分位距 | `1.5` |
| `gradient` | 距离变化梯度的指定百分位 | `95` |

在 `percentile` 模式下，较低阈值通常产生更多断点；这是相对分布的阈值，
**不是保留 65% 句子，也不是每块 65 个字符**。
距离大于阈值才形成候选断点，最终块数还受文本和其他参数影响。

中文若在句号后没有空格，课件的 `r"(?<=[。？！])\s+"` 就匹配不到那里。
按「标点后允许零个空白」拆句时，可考虑 `r"(?<=[。？！])\s*"`，并先检查实际拆句结果。

**与课件文字的差异**：当前本地 `langchain-experimental==0.4.1` 已实现 `gradient`，
不需要因为课件写了「需自定义实现」就另写一套。
语义分块也不保证主题绝对完整或块长不超模型限制，本次不调用外部嵌入接口演示它。

## 5.4 HTML、代码、Markdown：知道返回什么

**HTML**：`headers_to_split_on=[("h1", "标题1"), ("h2", "标题2")]` 指定层级；
输出 Document 的 metadata 可以保留所属标题，帮助检索时补充上下文。

**代码**：课件实际使用 `RecursiveCharacterTextSplitter.from_language(language=Language.PYTHON, ...)`。
它优先尝试类、函数等规则边界；块太长时仍会细分，所以不能保证永远不切开函数。
不是要另外导入一个名为 `CodeTextSplitter` 的通用类。

**Markdown**：课件混用了两类说明，需要区分：

- `MarkdownTextSplitter`：使用 Markdown 分隔规则做递归文本切分。
- `MarkdownHeaderTextSplitter`：按 `#`、`##` 等标题组织内容与元数据。

不需要照抄修改 `splitter._is_separator_regex` 之类私有属性的写法；
先选择与需求相符的公开接口。标题拆分后仍然过长，可以再交给递归拆分器。

## 5.5 选用顺序

先看资料结构：普通文本优先试递归拆分；HTML / Markdown 需要标题上下文时先按标题组织；
代码使用语言规则。存在严格输入预算时再结合匹配的 Tokenizer；
只有确实需要主题边界、并能接受嵌入开销时，再评估语义分块。

## 笔记导航

1. [7_1_RAG原理与流程](7_1_RAG原理与流程.ipynb)
2. [7_2_Document与文档加载](7_2_Document与文档加载.ipynb)
3. [7_3_TextSplitter源码与Character切分](7_3_TextSplitter源码与Character切分.ipynb)
4. [7_4_RecursiveCharacterTextSplitter详解](7_4_RecursiveCharacterTextSplitter详解.ipynb)
5. [7_5_其他拆分方式速记](7_5_其他拆分方式速记.ipynb)
6. [7_6_Embedding与向量存储](7_6_Embedding与向量存储.ipynb)